## CONCEPT-RELATIONSHIP

In [1]:
import pandas as pd

In [4]:
df = pd.read_csv('/Users/rose/Desktop/KG-for-Clinical-Guideline/data/CONCEPT_RELATIONSHIP.csv', sep='\t')
df.head()

,concept_id_1,concept_id_2,relationship_id,valid_start_date,valid_end_date,invalid_reason
0,4210800,40642538,Has status,20220128,20991231,NaN
1,4210800,40642539,Has Module,20220128,20991231,NaN
2,42535514,40642537,Has status,20220128,20991231,NaN
3,42535514,40642539,Has Module,20220128,20991231,NaN
4,40530649,40642538,Has status,20220128,20991231,NaN


In [6]:
df['relationship_id'].value_counts()

relationship_id
Mapped from             4451072
Maps to                 4451072
Has marketed form       2000405
Marketed form of        2000405
RxNorm has dose form    1797125
                         ...   
Surface texture of            4
Surf character of             4
Has surface char              4
After                         1
Before                        1
Name: count, Length: 326, dtype: int64

In [17]:
# Maps to 관계를 가진 모든 행 필터링
maps_to_df = df[df['relationship_id'] == 'Maps to']
maps_to_df.head()

,concept_id_1,concept_id_2,relationship_id,valid_start_date,valid_end_date,invalid_reason
49956,45144491,42902856,Maps to,20180420,20991231,NaN
49971,45265026,19034832,Maps to,19700101,20991231,NaN
49972,45185309,19008026,Maps to,19700101,20991231,NaN
49973,45264994,19033377,Maps to,19700101,20991231,NaN
49974,45034816,715301,Maps to,20080602,20991231,NaN


In [11]:
df['relationship_id'].unique()

array(['Has status', 'Has Module', 'Plays role', 'Has due to', 'Is a',
       'Has focus', 'Has interprets', 'Has indir proc site',
       'Has brand name', 'Has finding site', 'Has clinical course',
       'Has interpretation', 'Has scale type', 'Has inherent loc',
       'Brand name of', 'Has occurrence', 'Has asso morph',
       'Has dir proc site', 'Has comp material', 'Is sterile',
       'Has absorbability', 'Has method', 'Has active ing',
       'Has temporal context', 'Has relat context', 'Followed by',
       'Inheres in', 'Has property', 'Has technique', 'Has disposition',
       'Has pathology', 'Has coating material', 'Has causative agent',
       'Has surgical appr', 'Has access', 'Maps to', 'Mapped from',
       'Finding asso with', 'Has dir device', 'Has laterality',
       'Has specimen topo', 'Has specimen subst', 'Has direct site',
       'Has dir morph', 'Has asso finding', 'SPL - RxNorm',
       'RxNorm - SPL', 'RxNorm has ing', 'Contains',
       'RxNorm has dose f

In [16]:
df[df['concept_id_1'] == 4078821]

,concept_id_1,concept_id_2,relationship_id,valid_start_date,valid_end_date,invalid_reason
7278337,4078821,40642538,Has status,20220128,20991231,NaN
7278338,4078821,40642539,Has Module,20220128,20991231,NaN
17089412,4078821,4311405,Concept replaced by,20220930,20991231,NaN
17131515,4078821,4311405,Maps to,20230927,20991231,NaN


## Elasticsearch

In [4]:
import os
ES_SERVER_HOST = os.getenv("ES_SERVER_HOST", "3.35.110.161")
ES_SERVER_PORT = int(os.getenv("ES_SERVER_PORT", "9200"))
ES_SERVER_USERNAME = os.getenv("ES_SERVER_USERNAME", "elastic")
ES_SERVER_PASSWORD = os.getenv("ES_SERVER_PASSWORD", "snomed")

from elasticsearch import Elasticsearch

# ES 8.x 문법 (기본 권장)
es = Elasticsearch(
    f"http://{ES_SERVER_HOST}:{ES_SERVER_PORT}",
    basic_auth=(ES_SERVER_USERNAME, ES_SERVER_PASSWORD),
    request_timeout=60,
)

In [13]:
# 연결 확인
info = es.info()
print("Cluster name:", info.get('cluster_name'))
print("Version:", info.get('version', {}).get('number'))

# 인덱스 확인
indices = sorted(es.indices.get(index="*").keys())
print("\n=== 전체 인덱스 목록 ===")
for idx, index in enumerate(indices, 1):
    print(f"{idx:2d}. {index}")
print(f"\n총 인덱스 수: {len(indices)}")

Cluster name: docker-cluster
Version: 9.0.0

=== 전체 인덱스 목록 ===
 1. concept-condition
 2. concept-condition-device
 3. concept-condition-meas
 4. concept-condition_status
 5. concept-cost
 6. concept-currency
 7. concept-device
 8. concept-drug
 9. concept-episode
10. concept-ethnicity
11. concept-gender
12. concept-geography
13. concept-language
14. concept-meas_value
15. concept-meas_value_operator
16. concept-measurement
17. concept-metadata
18. concept-note
19. concept-observation
20. concept-payer
21. concept-plan
22. concept-plan_stop_reason
23. concept-procedure
24. concept-provider
25. concept-race
26. concept-relationship
27. concept-revenue_code
28. concept-route
29. concept-spec_anatomic_site
30. concept-specimen
31. concept-sponsor
32. concept-type_concept
33. concept-unit
34. concept-visit

총 인덱스 수: 34


In [15]:
import pandas as pd

rows = es.cat.indices(format="json", s="index", bytes="mb")
df = pd.DataFrame(rows)[["index", "docs.count", "store.size"]]
df.rename(columns={"index":"index_name","docs.count":"docs","store.size":"store_mb"}, inplace=True)
df

,index_name,docs,store_mb
0,concept-condition,268040,70
1,concept-condition-device,1,0
2,concept-condition-meas,8,0
3,concept-condition_status,22,0
4,concept-cost,51,0
5,concept-currency,180,0
6,concept-device,237173,56
7,concept-drug,4772484,1292
8,concept-episode,18,0
9,concept-ethnicity,2,0


In [12]:
q = {
    "query": {
        "query_string": {
            "query": "Acute coronary syndrome",
            "fields": ["*"],
            "analyze_wildcard": True,
            "default_operator": "AND"
        }
    },
    "_source": False,
    "size": 1
}

resp = es.search(index="concept-condition", body=q)
found = resp["hits"]["total"]["value"] > 0 if isinstance(resp["hits"]["total"], dict) else resp["hits"]["total"] > 0
print("ACS 존재 여부:", found)

ACS 존재 여부: True


## CONCEPT.csv

In [2]:
df = pd.read_csv('/Users/rose/Desktop/KG-for-Clinical-Guideline/data/CONCEPT.csv', sep='\t')
df.head()

/var/folders/km/l4y95bgj6lv844p6y3h17gcc0000gn/T/ipykernel_21259/1333313907.py:1: DtypeWarning: Columns (5,6,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/rose/Desktop/KG-for-Clinical-Guideline/data/CONCEPT.csv', sep='\t')


,concept_id,concept_name,domain_id,vocabulary_id,concept_class_id,standard_concept,concept_code,valid_start_date,valid_end_date,invalid_reason
0,45756805,Pediatric Cardiology,Provider,ABMS,Physician Specialty,S,OMOP4821938,19700101,20991231,NaN
1,45756804,Pediatric Anesthesiology,Provider,ABMS,Physician Specialty,S,OMOP4821939,19700101,20991231,NaN
2,45756803,Pathology-Anatomic / Pathology-Clinical,Provider,ABMS,Physician Specialty,S,OMOP4821940,19700101,20991231,NaN
3,45756802,Pathology - Pediatric,Provider,ABMS,Physician Specialty,S,OMOP4821941,19700101,20991231,NaN
4,45756801,Pathology - Molecular Genetic,Provider,ABMS,Physician Specialty,S,OMOP4821942,19700101,20991231,NaN
